# SVHN Digit Recognition — ANN vs CNN

MIT ADSP Elective — Deep Learning. Low-code lab notebook.

This notebook trains two Artificial Neural Networks (ANN-1, ANN-2) and two
Convolutional Neural Networks (CNN-1, CNN-2) on the SVHN single-digit grayscale
dataset, and writes all facts and figures needed for the business report to
`results/results_pack.md` and `report_figs/`.

All "Observations" headings below are intentionally left blank for the report author.


## 0. Setup, Seeds, and Device Check

In [1]:
import os

# Suppress TensorFlow C++ INFO/WARNING log lines (not Python warnings).
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '1'
os.environ.setdefault('MPLCONFIGDIR', os.path.join(os.getcwd(), '..', '.mplcache'))

import random
import time
import json
from pathlib import Path

import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input, Dense, Dropout, BatchNormalization, LeakyReLU,
    Conv2D, MaxPooling2D, Flatten,
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import confusion_matrix, classification_report

SEED = 42


def set_all_seeds(seed=SEED):
    """Re-seed Python, NumPy, and TensorFlow before every model build."""
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    keras.utils.set_random_seed(seed)


set_all_seeds(SEED)

REPO_ROOT = Path('..').resolve()
DATA_PATH = REPO_ROOT / 'data' / 'SVHN_single_grey1.h5'
FIGS_DIR = REPO_ROOT / 'report_figs'
RESULTS_DIR = REPO_ROOT / 'results'
FIGS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('TensorFlow version:', tf.__version__)
print('Keras version:', keras.__version__)
gpu_devices = tf.config.list_physical_devices('GPU')
print('GPU devices visible to TensorFlow:', gpu_devices)
DEVICE_INFO = f"GPU: {gpu_devices}" if gpu_devices else "CPU only (no GPU/Metal device found)"
print('Device that will be used for training:', DEVICE_INFO)


TensorFlow version: 2.19.1
Keras version: 3.15.1
GPU devices visible to TensorFlow: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Device that will be used for training: GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 1. Load Data

The file `SVHN_single_grey1.h5` ships with three pre-split arrays: `X_train`/`y_train`
(42,000 images), `X_val`/`y_val` (60,000 images), and `X_test`/`y_test` (18,000 images).
Per the standard convention for this course dataset, only the train and test arrays are
used; the shipped validation split is not loaded. Validation accuracy for every model
below comes from a `validation_split=0.2` carve-out of the training data at fit time.

In [2]:
with h5py.File(DATA_PATH, 'r') as f:
    print('Keys in file:', list(f.keys()))
    X_train_raw = f['X_train'][:]
    y_train_raw = f['y_train'][:]
    X_test_raw = f['X_test'][:]
    y_test_raw = f['y_test'][:]

print('X_train_raw shape:', X_train_raw.shape, X_train_raw.dtype)
print('y_train_raw shape:', y_train_raw.shape, y_train_raw.dtype)
print('X_test_raw shape:', X_test_raw.shape, X_test_raw.dtype)
print('y_test_raw shape:', y_test_raw.shape, y_test_raw.dtype)
print('Label classes (train):', np.unique(y_train_raw))
print('Label classes (test):', np.unique(y_test_raw))
print('Pixel min/max (train):', X_train_raw.min(), X_train_raw.max())
print('Pixel min/max (test):', X_test_raw.min(), X_test_raw.max())

NEEDS_SCALING = X_train_raw.max() > 1.0
print('Pixels already in [0, 1]:', not NEEDS_SCALING)


Keys in file: ['X_test', 'X_train', 'X_val', 'y_test', 'y_train', 'y_val']
X_train_raw shape: (42000, 32, 32) float32
y_train_raw shape: (42000,) uint8
X_test_raw shape: (18000, 32, 32) float32
y_test_raw shape: (18000,) uint8
Label classes (train): [0 1 2 3 4 5 6 7 8 9]
Label classes (test): [0 1 2 3 4 5 6 7 8 9]
Pixel min/max (train): 0.0 254.9745
Pixel min/max (test): 0.0 254.9745
Pixels already in [0, 1]: False


## 2. Data Overview (facts for the report)

In [3]:
IMAGE_SIZE = X_train_raw.shape[1:]
N_TRAIN, N_TEST = X_train_raw.shape[0], X_test_raw.shape[0]
N_CLASSES = 10

print(f'Train images: {N_TRAIN}, Test images: {N_TEST}, Image size: {IMAGE_SIZE}')

# Sample image grid
rng = np.random.RandomState(SEED)
sample_idx = rng.choice(N_TRAIN, size=25, replace=False)
fig, axes = plt.subplots(5, 5, figsize=(8, 8))
for ax, idx in zip(axes.flat, sample_idx):
    ax.imshow(X_train_raw[idx], cmap='gray')
    ax.set_title(f'label={y_train_raw[idx]}', fontsize=9)
    ax.axis('off')
fig.suptitle('Sample SVHN training images')
fig.tight_layout()
fig.savefig(FIGS_DIR / 'sample_images.png', dpi=150)
plt.close(fig)
print('Saved report_figs/sample_images.png')

# Class distribution
train_counts = pd.Series(y_train_raw).value_counts().sort_index()
test_counts = pd.Series(y_test_raw).value_counts().sort_index()
print('Train class counts:')
print(train_counts)
print('Test class counts:')
print(test_counts)

fig, ax = plt.subplots(figsize=(8, 5))
width = 0.4
x = np.arange(N_CLASSES)
ax.bar(x - width / 2, train_counts.values, width=width, label='train')
ax.bar(x + width / 2, test_counts.values, width=width, label='test')
ax.set_xticks(x)
ax.set_xlabel('Digit class')
ax.set_ylabel('Count')
ax.set_title('Class distribution: train vs test')
ax.legend()
fig.tight_layout()
fig.savefig(FIGS_DIR / 'class_distribution.png', dpi=150)
plt.close(fig)
print('Saved report_figs/class_distribution.png')


Train images: 42000, Test images: 18000, Image size: (32, 32)


Saved report_figs/sample_images.png
Train class counts:
0    4186
1    4172
2    4197
3    4281
4    4188
5    4232
6    4168
7    4192
8    4188
9    4196
Name: count, dtype: int64
Test class counts:
0    1814
1    1828
2    1803
3    1719
4    1812
5    1768
6    1832
7    1808
8    1812
9    1804
Name: count, dtype: int64
Saved report_figs/class_distribution.png


# Part 1 — Artificial Neural Networks (ANN)

## 1.1 Data Preprocessing for ANN

Steps: flatten each 32x32 image to a 1,024-length vector, normalize pixel values to
the 0-1 range, and one-hot encode the 10 digit-class labels.

In [4]:
print('--- Before ANN preprocessing ---')
print('X_train_raw:', X_train_raw.shape, 'y_train_raw:', y_train_raw.shape)
print('X_test_raw:', X_test_raw.shape, 'y_test_raw:', y_test_raw.shape)

# Flatten 32x32 -> 1024
X_train_ann = X_train_raw.reshape(N_TRAIN, 32 * 32).astype('float32')
X_test_ann = X_test_raw.reshape(N_TEST, 32 * 32).astype('float32')
print('After flatten -> X_train_ann:', X_train_ann.shape, 'X_test_ann:', X_test_ann.shape)

# Normalize to 0-1
if NEEDS_SCALING:
    X_train_ann = X_train_ann / 255.0
    X_test_ann = X_test_ann / 255.0
print('After normalize -> min/max (train):', X_train_ann.min(), X_train_ann.max())
print('After normalize -> min/max (test):', X_test_ann.min(), X_test_ann.max())

# One-hot encode labels (10 classes)
y_train_ann = to_categorical(y_train_raw, num_classes=N_CLASSES)
y_test_ann = to_categorical(y_test_raw, num_classes=N_CLASSES)
print('After one-hot -> y_train_ann:', y_train_ann.shape, 'y_test_ann:', y_test_ann.shape)


--- Before ANN preprocessing ---
X_train_raw: (42000, 32, 32) y_train_raw: (42000,)
X_test_raw: (18000, 32, 32) y_test_raw: (18000,)
After flatten -> X_train_ann: (42000, 1024) X_test_ann: (18000, 1024)


After normalize -> min/max (train): 0.0 0.9999
After normalize -> min/max (test): 0.0 0.9999
After one-hot -> y_train_ann: (42000, 10) y_test_ann: (18000, 10)


## 1.2 Training Helper

Shared training settings for every model in this notebook: Adam optimizer
(`learning_rate=0.001`), `categorical_crossentropy` loss, accuracy metric,
20 epochs, batch size 128, `validation_split=0.2`, no early stopping.

Before every real 20-epoch run, a throwaway 1-epoch fit on a freshly built model
estimates the total training time. That model is discarded and the seeds are reset
before the real run.

In [5]:
EPOCHS = 20
BATCH_SIZE = 128
VALIDATION_SPLIT = 0.2


def make_optimizer():
    return Adam(learning_rate=0.001)


def estimate_and_train(build_fn, X, y, model_name):
    """Run a 1-epoch throwaway fit to estimate time, then the real 20-epoch fit."""
    set_all_seeds(SEED)
    throwaway_model = build_fn()
    t0 = time.time()
    throwaway_model.fit(
        X, y, epochs=1, batch_size=BATCH_SIZE,
        validation_split=VALIDATION_SPLIT, verbose=0,
    )
    per_epoch_estimate = time.time() - t0
    total_estimate_min = per_epoch_estimate * EPOCHS / 60.0
    print(f'[{model_name}] Estimated time/epoch: {per_epoch_estimate:.2f}s -> '
          f'~{total_estimate_min:.1f} min for {EPOCHS} epochs.')
    del throwaway_model

    set_all_seeds(SEED)
    model = build_fn()
    t0 = time.time()
    history = model.fit(
        X, y, epochs=EPOCHS, batch_size=BATCH_SIZE,
        validation_split=VALIDATION_SPLIT, verbose=2,
    )
    train_time_sec = time.time() - t0
    print(f'[{model_name}] Actual training time: {train_time_sec:.1f}s '
          f'({train_time_sec / 60.0:.2f} min) for {EPOCHS} epochs.')
    return model, history, train_time_sec


def plot_accuracy_curve(history, model_name, save_path):
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(history.history['accuracy'], label='train accuracy')
    ax.plot(history.history['val_accuracy'], label='validation accuracy')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy')
    ax.set_title(f'{model_name}: Train vs Validation Accuracy')
    ax.legend()
    fig.tight_layout()
    fig.savefig(save_path, dpi=150)
    plt.close(fig)
    print(f'Saved {save_path}')


def summary_to_string(model):
    lines = []
    model.summary(print_fn=lambda line: lines.append(line))
    return '\n'.join(lines)


model_facts = {}


## 1.3 ANN-1: Baseline Fully-Connected Network

Architecture: Dense(256, ReLU) -> Dense(128, ReLU) -> Dense(64, ReLU) -> Dense(10, softmax).

In [6]:
def build_ann1():
    model = Sequential([
        Input(shape=(1024,)),
        Dense(256, activation='relu'),
        Dense(128, activation='relu'),
        Dense(64, activation='relu'),
        Dense(10, activation='softmax'),
    ], name='ANN_1_baseline')
    model.compile(optimizer=make_optimizer(), loss='categorical_crossentropy', metrics=['accuracy'])
    return model


ann1_preview = build_ann1()
ann1_preview.summary()
ANN1_SUMMARY = summary_to_string(ann1_preview)
ANN1_PARAMS = ann1_preview.count_params()
del ann1_preview


I0000 00:00:1791848826.509350 10364892 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1791848826.509559 10364892 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Model: "ANN_1_baseline"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 256)            │       262,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 304,202 (1.16 MB)

 Trainable params: 304,202 (1.16 MB)

 Non-trainable params: 0 (0.00 B)

In [7]:
ann1_model, ann1_history, ann1_time = estimate_and_train(build_ann1, X_train_ann, y_train_ann, 'ANN-1')


[ANN-1] Estimated time/epoch: 6.90s -> ~2.3 min for 20 epochs.
Epoch 1/20


263/263 - 5s - 18ms/step - accuracy: 0.1028 - loss: 2.5673 - val_accuracy: 0.0861 - val_loss: 2.3895


Epoch 2/20


263/263 - 4s - 15ms/step - accuracy: 0.0995 - loss: 2.3913 - val_accuracy: 0.1021 - val_loss: 2.3781


Epoch 3/20


...

Epoch 19/20


263/263 - 4s - 15ms/step - accuracy: 0.1033 - loss: 98.2795 - val_accuracy: 0.1112 - val_loss: 123.7978


Epoch 20/20


263/263 - 4s - 16ms/step - accuracy: 0.1020 - loss: 104.5171 - val_accuracy: 0.1067 - val_loss: 140.1229


[ANN-1] Actual training time: 79.3s (1.32 min) for 20 epochs.


In [8]:
ann1_train_acc = ann1_history.history['accuracy'][-1]
ann1_val_acc = ann1_history.history['val_accuracy'][-1]
ann1_test_loss, ann1_test_acc = ann1_model.evaluate(X_test_ann, y_test_ann, verbose=0)
ann1_epochs_completed = len(ann1_history.history['accuracy'])

print(f'ANN-1 train accuracy: {ann1_train_acc:.4f}')
print(f'ANN-1 validation accuracy: {ann1_val_acc:.4f}')
print(f'ANN-1 test accuracy: {ann1_test_acc:.4f}')
print(f'ANN-1 epochs completed: {ann1_epochs_completed}')

plot_accuracy_curve(ann1_history, 'ANN-1', FIGS_DIR / 'ann1_accuracy_curve.png')

model_facts['ANN-1'] = {
    'summary': ANN1_SUMMARY,
    'params': ANN1_PARAMS,
    'train_acc': ann1_train_acc,
    'val_acc': ann1_val_acc,
    'test_acc': ann1_test_acc,
    'epochs': ann1_epochs_completed,
    'train_time_sec': ann1_time,
}


ANN-1 train accuracy: 0.1020
ANN-1 validation accuracy: 0.1067
ANN-1 test accuracy: 0.1047
ANN-1 epochs completed: 20
Saved /Users/cviscont/Cyberlabs/Web Apps/mit-SVHN deep learning/report_figs/ann1_accuracy_curve.png


#### Observations

## 1.4 ANN-2: Deeper Network with BatchNorm, LeakyReLU, and Dropout

Architecture: Dense(512) -> BatchNorm -> LeakyReLU(0.1) -> Dropout(0.2) ->
Dense(256) -> BatchNorm -> LeakyReLU(0.1) -> Dropout(0.2) ->
Dense(128) -> BatchNorm -> LeakyReLU(0.1) -> Dropout(0.2) ->
Dense(64) -> LeakyReLU(0.1) -> Dense(10, softmax).

In [9]:
def build_ann2():
    model = Sequential([
        Input(shape=(1024,)),
        Dense(512),
        BatchNormalization(),
        LeakyReLU(negative_slope=0.1),
        Dropout(0.2),
        Dense(256),
        BatchNormalization(),
        LeakyReLU(negative_slope=0.1),
        Dropout(0.2),
        Dense(128),
        BatchNormalization(),
        LeakyReLU(negative_slope=0.1),
        Dropout(0.2),
        Dense(64),
        LeakyReLU(negative_slope=0.1),
        Dense(10, activation='softmax'),
    ], name='ANN_2_deeper')
    model.compile(optimizer=make_optimizer(), loss='categorical_crossentropy', metrics=['accuracy'])
    return model


ann2_preview = build_ann2()
ann2_preview.summary()
ANN2_SUMMARY = summary_to_string(ann2_preview)
ANN2_PARAMS = ann2_preview.count_params()
del ann2_preview


Model: "ANN_2_deeper"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_12 (Dense)                │ (None, 512)            │       524,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu (LeakyReLU)         │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_1 (LeakyReLU)       │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_2 (LeakyReLU)       │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_3 (LeakyReLU)       │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 701,514 (2.68 MB)

 Trainable params: 699,722 (2.67 MB)

 Non-trainable params: 1,792 (7.00 KB)

In [10]:
ann2_model, ann2_history, ann2_time = estimate_and_train(build_ann2, X_train_ann, y_train_ann, 'ANN-2')


[ANN-2] Estimated time/epoch: 9.54s -> ~3.2 min for 20 epochs.
Epoch 1/20


263/263 - 9s - 34ms/step - accuracy: 0.4319 - loss: 1.6608 - val_accuracy: 0.4725 - val_loss: 1.6146


Epoch 2/20


263/263 - 7s - 27ms/step - accuracy: 0.6427 - loss: 1.1230 - val_accuracy: 0.5742 - val_loss: 1.3082


Epoch 3/20


...

Epoch 19/20


263/263 - 7s - 27ms/step - accuracy: 0.8354 - loss: 0.5075 - val_accuracy: 0.7033 - val_loss: 0.8947


Epoch 20/20


263/263 - 7s - 26ms/step - accuracy: 0.8385 - loss: 0.4997 - val_accuracy: 0.6889 - val_loss: 0.9373


[ANN-2] Actual training time: 142.8s (2.38 min) for 20 epochs.


In [11]:
ann2_train_acc = ann2_history.history['accuracy'][-1]
ann2_val_acc = ann2_history.history['val_accuracy'][-1]
ann2_test_loss, ann2_test_acc = ann2_model.evaluate(X_test_ann, y_test_ann, verbose=0)
ann2_epochs_completed = len(ann2_history.history['accuracy'])

print(f'ANN-2 train accuracy: {ann2_train_acc:.4f}')
print(f'ANN-2 validation accuracy: {ann2_val_acc:.4f}')
print(f'ANN-2 test accuracy: {ann2_test_acc:.4f}')
print(f'ANN-2 epochs completed: {ann2_epochs_completed}')

plot_accuracy_curve(ann2_history, 'ANN-2', FIGS_DIR / 'ann2_accuracy_curve.png')

model_facts['ANN-2'] = {
    'summary': ANN2_SUMMARY,
    'params': ANN2_PARAMS,
    'train_acc': ann2_train_acc,
    'val_acc': ann2_val_acc,
    'test_acc': ann2_test_acc,
    'epochs': ann2_epochs_completed,
    'train_time_sec': ann2_time,
}


ANN-2 train accuracy: 0.8385
ANN-2 validation accuracy: 0.6889
ANN-2 test accuracy: 0.6877
ANN-2 epochs completed: 20
Saved /Users/cviscont/Cyberlabs/Web Apps/mit-SVHN deep learning/report_figs/ann2_accuracy_curve.png


#### Observations

## 1.5 Main Differences: ANN-1 vs ANN-2 (facts only)

In [12]:
ann_diff_facts = {
    'ANN-1 layers': '3 Dense (256, 128, 64) + output, all ReLU',
    'ANN-2 layers': '4 Dense (512, 256, 128, 64) + output, LeakyReLU(0.1)',
    'ANN-1 Dropout': 'none',
    'ANN-2 Dropout': 'Dropout(0.2) after first three Dense blocks',
    'ANN-1 BatchNorm': 'none',
    'ANN-2 BatchNorm': 'after Dense(512), Dense(256), Dense(128)',
    'ANN-1 activation': 'ReLU',
    'ANN-2 activation': 'LeakyReLU(negative_slope=0.1)',
    'ANN-1 parameter count': ANN1_PARAMS,
    'ANN-2 parameter count': ANN2_PARAMS,
}
for k, v in ann_diff_facts.items():
    print(f'{k}: {v}')


ANN-1 layers: 3 Dense (256, 128, 64) + output, all ReLU
ANN-2 layers: 4 Dense (512, 256, 128, 64) + output, LeakyReLU(0.1)
ANN-1 Dropout: none
ANN-2 Dropout: Dropout(0.2) after first three Dense blocks
ANN-1 BatchNorm: none
ANN-2 BatchNorm: after Dense(512), Dense(256), Dense(128)
ANN-1 activation: ReLU
ANN-2 activation: LeakyReLU(negative_slope=0.1)
ANN-1 parameter count: 304202
ANN-2 parameter count: 701514


## 1.6 Write Partial Results Pack (Pass 1 checkpoint)

Writes `results/results_pack.md` with the data overview and ANN sections populated.
The CNN sections are written as "pending — Pass 2" placeholders and will be
overwritten with full content once Pass 2 runs.

In [13]:
def fmt_pct(x):
    return f'{x * 100:.2f}%'


def write_results_pack(cnn_section_md='_Pending — Pass 2 not yet run._\n',
                        comparison_section_md='_Pending — Pass 2 not yet run._\n',
                        diagnostics_section_md='_Pending — Pass 2 not yet run._\n',
                        facts_section_md='_Pending — Pass 2 not yet run._\n'):
    lines = []
    lines.append('# SVHN Digit Recognition — Results Pack\n')
    lines.append(f'Device used for training: `{DEVICE_INFO}`\n')
    lines.append(f'Seed: {SEED} (NumPy, Python random, TensorFlow/Keras). '
                  'Note: Metal GPU execution is not guaranteed to be bit-for-bit '
                  'deterministic even with fixed seeds.\n')

    lines.append('## 1. Data Overview\n')
    lines.append(f'- Train shape: {X_train_raw.shape}, dtype {X_train_raw.dtype}')
    lines.append(f'- Test shape: {X_test_raw.shape}, dtype {X_test_raw.dtype}')
    lines.append(f'- Image size: {IMAGE_SIZE[0]}x{IMAGE_SIZE[1]} grayscale')
    lines.append(f'- Classes: {N_CLASSES} (digits 0-9)')
    lines.append('- Sample image grid: `report_figs/sample_images.png`')
    lines.append('- Class distribution bar chart: `report_figs/class_distribution.png`')
    lines.append('- Train class counts: ' + ', '.join(f'{i}:{c}' for i, c in train_counts.items()))
    lines.append('- Test class counts: ' + ', '.join(f'{i}:{c}' for i, c in test_counts.items()))
    lines.append('')

    lines.append('## 2. Model Performance\n')
    lines.append('### ANN-1\n')
    lines.append('```')
    lines.append(model_facts['ANN-1']['summary'])
    lines.append('```')
    lines.append(f"- Parameters: {model_facts['ANN-1']['params']:,}")
    lines.append(f"- Train accuracy: {fmt_pct(model_facts['ANN-1']['train_acc'])}")
    lines.append(f"- Validation accuracy: {fmt_pct(model_facts['ANN-1']['val_acc'])}")
    lines.append(f"- Test accuracy: {fmt_pct(model_facts['ANN-1']['test_acc'])}")
    lines.append(f"- Epochs: {model_facts['ANN-1']['epochs']}")
    lines.append(f"- Training time: {model_facts['ANN-1']['train_time_sec']:.1f}s")
    lines.append('- Accuracy curve: `report_figs/ann1_accuracy_curve.png`')
    lines.append('')
    lines.append('### ANN-2\n')
    lines.append('```')
    lines.append(model_facts['ANN-2']['summary'])
    lines.append('```')
    lines.append(f"- Parameters: {model_facts['ANN-2']['params']:,}")
    lines.append(f"- Train accuracy: {fmt_pct(model_facts['ANN-2']['train_acc'])}")
    lines.append(f"- Validation accuracy: {fmt_pct(model_facts['ANN-2']['val_acc'])}")
    lines.append(f"- Test accuracy: {fmt_pct(model_facts['ANN-2']['test_acc'])}")
    lines.append(f"- Epochs: {model_facts['ANN-2']['epochs']}")
    lines.append(f"- Training time: {model_facts['ANN-2']['train_time_sec']:.1f}s")
    lines.append('- Accuracy curve: `report_figs/ann2_accuracy_curve.png`')
    lines.append('')

    lines.append(cnn_section_md)

    lines.append('## 3. Main Differences\n')
    lines.append('### ANN-1 vs ANN-2\n')
    for k, v in ann_diff_facts.items():
        lines.append(f'- {k}: {v}')
    lines.append('')
    lines.append(comparison_section_md)

    lines.append('## 5-6. Best-Model Diagnostics\n')
    lines.append(diagnostics_section_md)

    lines.append('## 7. Model Comparison Table\n')
    lines.append('_(see comparison section above once Pass 2 completes)_\n')

    lines.append('## 8. Facts To Check\n')
    lines.append(facts_section_md)

    (RESULTS_DIR / 'results_pack.md').write_text('\n'.join(lines))
    print('Wrote results/results_pack.md')


write_results_pack()


Wrote results/results_pack.md


# Part 2 — Convolutional Neural Networks (CNN)

## 2.1 Data Preprocessing for CNN

Steps: reshape each 32x32 image to (32, 32, 1), normalize pixel values to the 0-1
range, and one-hot encode the 10 digit-class labels. CNN preprocessing starts from
the same raw arrays loaded in Section 1, independent of the ANN preprocessing above.

In [ ]:
print('--- Before CNN preprocessing ---')
print('X_train_raw:', X_train_raw.shape, 'y_train_raw:', y_train_raw.shape)
print('X_test_raw:', X_test_raw.shape, 'y_test_raw:', y_test_raw.shape)

# Reshape 32x32 -> (32, 32, 1)
X_train_cnn = X_train_raw.reshape(N_TRAIN, 32, 32, 1).astype('float32')
X_test_cnn = X_test_raw.reshape(N_TEST, 32, 32, 1).astype('float32')
print('After reshape -> X_train_cnn:', X_train_cnn.shape, 'X_test_cnn:', X_test_cnn.shape)

# Normalize to 0-1
if NEEDS_SCALING:
    X_train_cnn = X_train_cnn / 255.0
    X_test_cnn = X_test_cnn / 255.0
print('After normalize -> min/max (train):', X_train_cnn.min(), X_train_cnn.max())
print('After normalize -> min/max (test):', X_test_cnn.min(), X_test_cnn.max())

# One-hot encode labels (10 classes)
y_train_cnn = to_categorical(y_train_raw, num_classes=N_CLASSES)
y_test_cnn = to_categorical(y_test_raw, num_classes=N_CLASSES)
print('After one-hot -> y_train_cnn:', y_train_cnn.shape, 'y_test_cnn:', y_test_cnn.shape)


## 2.2 CNN-1: Baseline Convolutional Network

Architecture: Conv2D(32, 3x3, ReLU) -> MaxPool(2x2) -> Conv2D(64, 3x3, ReLU) ->
MaxPool(2x2) -> Flatten -> Dense(64, ReLU) -> Dense(10, softmax).

In [ ]:
def build_cnn1():
    model = Sequential([
        Input(shape=(32, 32, 1)),
        Conv2D(32, (3, 3), activation='relu'),
        MaxPooling2D((2, 2)),
        Conv2D(64, (3, 3), activation='relu'),
        MaxPooling2D((2, 2)),
        Flatten(),
        Dense(64, activation='relu'),
        Dense(10, activation='softmax'),
    ], name='CNN_1_baseline')
    model.compile(optimizer=make_optimizer(), loss='categorical_crossentropy', metrics=['accuracy'])
    return model


cnn1_preview = build_cnn1()
cnn1_preview.summary()
CNN1_SUMMARY = summary_to_string(cnn1_preview)
CNN1_PARAMS = cnn1_preview.count_params()
del cnn1_preview


In [ ]:
cnn1_model, cnn1_history, cnn1_time = estimate_and_train(build_cnn1, X_train_cnn, y_train_cnn, 'CNN-1')


In [ ]:
cnn1_train_acc = cnn1_history.history['accuracy'][-1]
cnn1_val_acc = cnn1_history.history['val_accuracy'][-1]
cnn1_test_loss, cnn1_test_acc = cnn1_model.evaluate(X_test_cnn, y_test_cnn, verbose=0)
cnn1_epochs_completed = len(cnn1_history.history['accuracy'])

print(f'CNN-1 train accuracy: {cnn1_train_acc:.4f}')
print(f'CNN-1 validation accuracy: {cnn1_val_acc:.4f}')
print(f'CNN-1 test accuracy: {cnn1_test_acc:.4f}')
print(f'CNN-1 epochs completed: {cnn1_epochs_completed}')

plot_accuracy_curve(cnn1_history, 'CNN-1', FIGS_DIR / 'cnn1_accuracy_curve.png')

model_facts['CNN-1'] = {
    'summary': CNN1_SUMMARY,
    'params': CNN1_PARAMS,
    'train_acc': cnn1_train_acc,
    'val_acc': cnn1_val_acc,
    'test_acc': cnn1_test_acc,
    'epochs': cnn1_epochs_completed,
    'train_time_sec': cnn1_time,
}


#### Observations

## 2.3 CNN-2: Deeper Convolutional Network with BatchNorm, LeakyReLU, and Dropout

Architecture: Conv2D(32) -> LeakyReLU(0.1) -> BatchNorm -> Conv2D(32) -> LeakyReLU(0.1)
-> MaxPool(2x2) -> Dropout(0.25) -> Conv2D(64) -> LeakyReLU(0.1) -> BatchNorm ->
Conv2D(64) -> LeakyReLU(0.1) -> MaxPool(2x2) -> Dropout(0.25) -> Flatten ->
Dense(128) -> LeakyReLU(0.1) -> Dropout(0.4) -> Dense(10, softmax).

In [ ]:
def build_cnn2():
    model = Sequential([
        Input(shape=(32, 32, 1)),
        Conv2D(32, (3, 3)),
        LeakyReLU(negative_slope=0.1),
        BatchNormalization(),
        Conv2D(32, (3, 3)),
        LeakyReLU(negative_slope=0.1),
        MaxPooling2D((2, 2)),
        Dropout(0.25),
        Conv2D(64, (3, 3)),
        LeakyReLU(negative_slope=0.1),
        BatchNormalization(),
        Conv2D(64, (3, 3)),
        LeakyReLU(negative_slope=0.1),
        MaxPooling2D((2, 2)),
        Dropout(0.25),
        Flatten(),
        Dense(128),
        LeakyReLU(negative_slope=0.1),
        Dropout(0.4),
        Dense(10, activation='softmax'),
    ], name='CNN_2_deeper')
    model.compile(optimizer=make_optimizer(), loss='categorical_crossentropy', metrics=['accuracy'])
    return model


cnn2_preview = build_cnn2()
cnn2_preview.summary()
CNN2_SUMMARY = summary_to_string(cnn2_preview)
CNN2_PARAMS = cnn2_preview.count_params()
del cnn2_preview


In [ ]:
cnn2_model, cnn2_history, cnn2_time = estimate_and_train(build_cnn2, X_train_cnn, y_train_cnn, 'CNN-2')


In [ ]:
cnn2_train_acc = cnn2_history.history['accuracy'][-1]
cnn2_val_acc = cnn2_history.history['val_accuracy'][-1]
cnn2_test_loss, cnn2_test_acc = cnn2_model.evaluate(X_test_cnn, y_test_cnn, verbose=0)
cnn2_epochs_completed = len(cnn2_history.history['accuracy'])

print(f'CNN-2 train accuracy: {cnn2_train_acc:.4f}')
print(f'CNN-2 validation accuracy: {cnn2_val_acc:.4f}')
print(f'CNN-2 test accuracy: {cnn2_test_acc:.4f}')
print(f'CNN-2 epochs completed: {cnn2_epochs_completed}')

plot_accuracy_curve(cnn2_history, 'CNN-2', FIGS_DIR / 'cnn2_accuracy_curve.png')

model_facts['CNN-2'] = {
    'summary': CNN2_SUMMARY,
    'params': CNN2_PARAMS,
    'train_acc': cnn2_train_acc,
    'val_acc': cnn2_val_acc,
    'test_acc': cnn2_test_acc,
    'epochs': cnn2_epochs_completed,
    'train_time_sec': cnn2_time,
}


#### Observations

## 2.4 Main Differences: CNN-1 vs CNN-2 (facts only)

In [ ]:
cnn_diff_facts = {
    'CNN-1 layers': '2 Conv2D (32, 64) + MaxPool each + Dense(64) + output, ReLU',
    'CNN-2 layers': '4 Conv2D (32, 32, 64, 64) + MaxPool x2 + Dense(128) + output, LeakyReLU(0.1)',
    'CNN-1 Dropout': 'none',
    'CNN-2 Dropout': 'Dropout(0.25) x2 (after each pooling block) + Dropout(0.4) before output',
    'CNN-1 BatchNorm': 'none',
    'CNN-2 BatchNorm': 'after first Conv2D in each of the two conv blocks',
    'CNN-1 activation': 'ReLU',
    'CNN-2 activation': 'LeakyReLU(negative_slope=0.1)',
    'CNN-1 parameter count': CNN1_PARAMS,
    'CNN-2 parameter count': CNN2_PARAMS,
}
for k, v in cnn_diff_facts.items():
    print(f'{k}: {v}')


## 3. Best-Model Diagnostics (ANN and CNN)

In [ ]:
def pick_best(name_a, name_b):
    """Best = highest test accuracy, then validation accuracy, then fewer params."""
    a, b = model_facts[name_a], model_facts[name_b]
    key = lambda d: (d['test_acc'], d['val_acc'], -d['params'])
    return name_a if key(a) >= key(b) else name_b


BEST_ANN = pick_best('ANN-1', 'ANN-2')
BEST_CNN = pick_best('CNN-1', 'CNN-2')
print('Best ANN by rule (test acc, then val acc, then fewer params):', BEST_ANN)
print('Best CNN by rule (test acc, then val acc, then fewer params):', BEST_CNN)

best_ann_model = {'ANN-1': ann1_model, 'ANN-2': ann2_model}[BEST_ANN]
best_cnn_model = {'CNN-1': cnn1_model, 'CNN-2': cnn2_model}[BEST_CNN]
best_ann_X_test, best_ann_y_test = X_test_ann, y_test_ann
best_cnn_X_test, best_cnn_y_test = X_test_cnn, y_test_cnn


In [ ]:
def confusion_and_report(model, X_test, y_test_oh, model_label, cm_path, cr_path):
    y_true = np.argmax(y_test_oh, axis=1)
    y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)

    cm = confusion_matrix(y_true, y_pred, labels=list(range(N_CLASSES)))
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=range(N_CLASSES), yticklabels=range(N_CLASSES))
    ax.set_xlabel('Predicted label')
    ax.set_ylabel('True label')
    ax.set_title(f'Confusion Matrix: {model_label}')
    fig.tight_layout()
    fig.savefig(cm_path, dpi=150)
    plt.close(fig)
    print(f'Saved {cm_path}')

    report_dict = classification_report(y_true, y_pred, digits=4, output_dict=True)
    report_str = classification_report(y_true, y_pred, digits=4)
    print(f'Classification report: {model_label}')
    print(report_str)

    report_df = pd.DataFrame(report_dict).transpose()
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.axis('off')
    tbl = ax.table(
        cellText=np.round(report_df.values, 4),
        rowLabels=report_df.index,
        colLabels=report_df.columns,
        loc='center',
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(8)
    ax.set_title(f'Classification Report: {model_label}', pad=20)
    fig.tight_layout()
    fig.savefig(cr_path, dpi=150)
    plt.close(fig)
    print(f'Saved {cr_path}')

    # Top 5 most-confused (true, pred) off-diagonal pairs
    cm_copy = cm.copy()
    np.fill_diagonal(cm_copy, 0)
    flat_idx = np.argsort(cm_copy, axis=None)[::-1][:5]
    pairs = []
    for idx in flat_idx:
        true_cls, pred_cls = np.unravel_index(idx, cm_copy.shape)
        count = cm_copy[true_cls, pred_cls]
        if count > 0:
            pairs.append((int(true_cls), int(pred_cls), int(count)))
    print(f'Top confused pairs ({model_label}):', pairs)

    return cm, report_str, pairs


ann_cm, ann_report_str, ann_top_pairs = confusion_and_report(
    best_ann_model, best_ann_X_test, best_ann_y_test, f'Best ANN ({BEST_ANN})',
    FIGS_DIR / 'confusion_matrix_ann_best.png', FIGS_DIR / 'classification_report_ann_best.png',
)
cnn_cm, cnn_report_str, cnn_top_pairs = confusion_and_report(
    best_cnn_model, best_cnn_X_test, best_cnn_y_test, f'Best CNN ({BEST_CNN})',
    FIGS_DIR / 'confusion_matrix_cnn_best.png', FIGS_DIR / 'classification_report_cnn_best.png',
)


## 4. Model Comparison Table (all four models)

In [ ]:
comparison_df = pd.DataFrame([
    {
        'Model': name,
        'Parameters': model_facts[name]['params'],
        'Train Acc': model_facts[name]['train_acc'],
        'Val Acc': model_facts[name]['val_acc'],
        'Test Acc': model_facts[name]['test_acc'],
        'Epochs': model_facts[name]['epochs'],
        'Train Time (s)': round(model_facts[name]['train_time_sec'], 1),
    }
    for name in ['ANN-1', 'ANN-2', 'CNN-1', 'CNN-2']
])
comparison_df


## 5. Facts To Check (no interpretation)

In [ ]:
facts_to_check = [
    f"Label range confirmed as 0-9 for both train and test (np.unique output above).",
    f"Pixel scaling applied: {NEEDS_SCALING} (raw max was > 1, so divided by 255).",
    f"Device used for training: {DEVICE_INFO}.",
    f"Validation definition: validation_split=0.2 of the training array at fit() time "
    f"(not the shipped X_val/y_val arrays in the .h5 file).",
    f"Seed used: {SEED} for Python random, NumPy, and TensorFlow/Keras "
    f"(keras.utils.set_random_seed). Metal GPU execution is not guaranteed "
    f"bit-for-bit deterministic even with fixed seeds.",
    f"Epochs completed per model: ANN-1={model_facts['ANN-1']['epochs']}, "
    f"ANN-2={model_facts['ANN-2']['epochs']}, CNN-1={model_facts['CNN-1']['epochs']}, "
    f"CNN-2={model_facts['CNN-2']['epochs']} (no early stopping used).",
    f"Best-ANN selection rule: highest test accuracy, then validation accuracy, "
    f"then fewer parameters. Selected: {BEST_ANN}.",
    f"Best-CNN selection rule: same rule. Selected: {BEST_CNN}.",
    f"Batch size: {BATCH_SIZE} for all four models.",
    f"The .h5 file also ships an unused X_val/y_val split (60,000 images); "
    f"it is not used anywhere in this notebook.",
]
for fact in facts_to_check:
    print('-', fact)


## 6. Finalize Results Pack (Pass 2 — full pack)

Overwrites `results/results_pack.md` with the complete CNN section, the main
differences bullets, the comparison table, best-model diagnostics, and the facts list.

In [ ]:
cnn_section_lines = []
cnn_section_lines.append('### CNN-1\n')
cnn_section_lines.append('```')
cnn_section_lines.append(model_facts['CNN-1']['summary'])
cnn_section_lines.append('```')
cnn_section_lines.append(f"- Parameters: {model_facts['CNN-1']['params']:,}")
cnn_section_lines.append(f"- Train accuracy: {fmt_pct(model_facts['CNN-1']['train_acc'])}")
cnn_section_lines.append(f"- Validation accuracy: {fmt_pct(model_facts['CNN-1']['val_acc'])}")
cnn_section_lines.append(f"- Test accuracy: {fmt_pct(model_facts['CNN-1']['test_acc'])}")
cnn_section_lines.append(f"- Epochs: {model_facts['CNN-1']['epochs']}")
cnn_section_lines.append(f"- Training time: {model_facts['CNN-1']['train_time_sec']:.1f}s")
cnn_section_lines.append('- Accuracy curve: `report_figs/cnn1_accuracy_curve.png`')
cnn_section_lines.append('')
cnn_section_lines.append('### CNN-2\n')
cnn_section_lines.append('```')
cnn_section_lines.append(model_facts['CNN-2']['summary'])
cnn_section_lines.append('```')
cnn_section_lines.append(f"- Parameters: {model_facts['CNN-2']['params']:,}")
cnn_section_lines.append(f"- Train accuracy: {fmt_pct(model_facts['CNN-2']['train_acc'])}")
cnn_section_lines.append(f"- Validation accuracy: {fmt_pct(model_facts['CNN-2']['val_acc'])}")
cnn_section_lines.append(f"- Test accuracy: {fmt_pct(model_facts['CNN-2']['test_acc'])}")
cnn_section_lines.append(f"- Epochs: {model_facts['CNN-2']['epochs']}")
cnn_section_lines.append(f"- Training time: {model_facts['CNN-2']['train_time_sec']:.1f}s")
cnn_section_lines.append('- Accuracy curve: `report_figs/cnn2_accuracy_curve.png`')
cnn_section_lines.append('')
cnn_section_md = '\n'.join(cnn_section_lines)

comparison_lines = []
comparison_lines.append('### CNN-1 vs CNN-2\n')
for k, v in cnn_diff_facts.items():
    comparison_lines.append(f'- {k}: {v}')
comparison_lines.append('')
comparison_section_md = '\n'.join(comparison_lines)

diagnostics_lines = []
diagnostics_lines.append(f'### Best ANN: {BEST_ANN}\n')
diagnostics_lines.append('- Confusion matrix: `report_figs/confusion_matrix_ann_best.png`')
diagnostics_lines.append('- Classification report image: `report_figs/classification_report_ann_best.png`')
diagnostics_lines.append('```')
diagnostics_lines.append(ann_report_str)
diagnostics_lines.append('```')
diagnostics_lines.append(f'- Top 5 most-confused (true -> predicted, count) pairs: {ann_top_pairs}')
diagnostics_lines.append('')
diagnostics_lines.append(f'### Best CNN: {BEST_CNN}\n')
diagnostics_lines.append('- Confusion matrix: `report_figs/confusion_matrix_cnn_best.png`')
diagnostics_lines.append('- Classification report image: `report_figs/classification_report_cnn_best.png`')
diagnostics_lines.append('```')
diagnostics_lines.append(cnn_report_str)
diagnostics_lines.append('```')
diagnostics_lines.append(f'- Top 5 most-confused (true -> predicted, count) pairs: {cnn_top_pairs}')
diagnostics_lines.append('')
diagnostics_section_md = '\n'.join(diagnostics_lines)

facts_section_md = '\n'.join(f'- {fact}' for fact in facts_to_check) + '\n'

write_results_pack(
    cnn_section_md=cnn_section_md,
    comparison_section_md=comparison_section_md,
    diagnostics_section_md=diagnostics_section_md,
    facts_section_md=facts_section_md,
)

# Append the comparison table as markdown (pandas .to_markdown needs tabulate)
with open(RESULTS_DIR / 'results_pack.md', 'r') as f:
    content = f.read()
table_md = comparison_df.to_string(index=False)
content = content.replace(
    '_(see comparison section above once Pass 2 completes)_\n',
    '```\n' + table_md + '\n```\n',
)
with open(RESULTS_DIR / 'results_pack.md', 'w') as f:
    f.write(content)
print('Finalized results/results_pack.md with full CNN + comparison + diagnostics content.')


## End of Notebook